In [1]:
import pandas as pd
from surprise import Dataset
from surprise import Reader
from surprise import KNNWithMeans
from surprise.model_selection import GridSearchCV
from surprise import SVD
import mlflow

In [2]:
test_df = pd.read_csv("data/full_wine_dataset.csv")
test_df[test_df["product_name"] == "Pinot Noir"].iloc[1].href

'https://www.danmurphys.com.au/product/DM_53212/minchinbury-pinot-noir?isFromSearch=false&isPersonalised=false&pageName=category_page'

In [3]:
df = pd.read_csv("data/training_datasets/filtered_users_products_df.csv")

In [4]:
df

,product_href,user_href,username,rating,date,product_name,product_description,colour,type,vintage,state,region,winery,size,non_member_price,member_price,body,sweetness,leftover_words,type_as_list
0,https://www.danmurphys.com.au/product/DM_10266...,NaN,Ange,4,21/07/2021,Elgin's Crossing Cabernet Sauvignon,Elgin’s Crossing commemorates pioneering vigne...,Red,cabernet sauvignon,2020*,NaN,Coonawarra,NaN,750mL,23.29,NaN,NaN,Dry,[],['cabernet sauvignon']
1,https://www.danmurphys.com.au/product/DM_11021...,https://www.danmurphys.com.au/review/shopper/2...,Bazza,5,02/02/2021,Church Block Cabernet Sauvignon Shiraz Merlot,Church Block was the first wine the late Greg ...,Red,cabernet sauvignon merlot shiraz,2020,South Australia,McLaren Vale,Paul Smith & Paul Carpenter,750mL,17.95,16.15,Full Bodied,Dry,[],"['cabernet sauvignon', 'merlot', 'shiraz']"
2,https://www.danmurphys.com.au/product/DM_11021...,NaN,Wine lover,5,11/10/2020,Church Block Cabernet Sauvignon Shiraz Merlot,Church Block was the first wine the late Greg ...,Red,cabernet sauvignon merlot shiraz,2020,South Australia,McLaren Vale,Paul Smith & Paul Carpenter,750mL,17.95,16.15,Full Bodied,Dry,[],"['cabernet sauvignon', 'merlot', 'shiraz']"
3,https://www.danmurphys.com.au/product/DM_11021...,NaN,John,5,02/10/2020,Church Block Cabernet Sauvignon Shiraz Merlot,Church Block was the first wine the late Greg ...,Red,cabernet sauvignon merlot shiraz,2020,South Australia,McLaren Vale,Paul Smith & Paul Carpenter,750mL,17.95,16.15,Full Bodied,Dry,[],"['cabernet sauvignon', 'merlot', 'shiraz']"
4,https://www.danmurphys.com.au/product/DM_11021...,https://www.danmurphys.com.au/review/shopper/6...,Steve,5,04/07/2020,Church Block Cabernet Sauvignon Shiraz Merlot,Church Block was the first wine the late Greg ...,Red,cabernet sauvignon merlot shiraz,2020,South Australia,McLaren Vale,Paul Smith & Paul Carpenter,750mL,17.95,16.15,Full Bodied,Dry,[],"['cabernet sauvignon', 'merlot', 'shiraz']"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
665,https://www.danmurphys.com.au/product/DM_79521...,https://www.danmurphys.com.au/review/shopper/8...,Mad Bear,3,13/09/2018,Watervale Riesling,This wine from Vickery displays hallmark Water...,White,riesling,2022,South Australia,Clare Valley,John Vickery & Keeda Zilm,750mL,22.29,19.30,Light Bodied,Dry,[],['riesling']
666,https://www.danmurphys.com.au/product/DM_78771...,https://www.danmurphys.com.au/review/shopper/2...,Phil,4,18/06/2023,Coonawarra Series Chardonnay,Parker Coonawarra Estate was established in 19...,White,chardonnay,2021,South Australia,Coonawarra,NaN,750mL,24.29,18.00,NaN,Dry,[],['chardonnay']
667,https://www.danmurphys.com.au/product/DM_78771...,NaN,Wine lover,1,07/03/2024,Coonawarra Series Chardonnay,Parker Coonawarra Estate was established in 19...,White,chardonnay,2021,South Australia,Coonawarra,NaN,750mL,24.29,18.00,NaN,Dry,[],['chardonnay']
668,https://www.danmurphys.com.au/product/DM_78771...,NaN,Andy,1,13/01/2021,Coonawarra Series Chardonnay,Parker Coonawarra Estate was established in 19...,White,chardonnay,2021,South Australia,Coonawarra,NaN,750mL,24.29,18.00,NaN,Dry,[],['chardonnay']


### Modelling using surprise

In [5]:
reader = Reader(rating_scale=(1, 5))

# Loads Pandas dataframe
data = Dataset.load_from_df(df[["username", "product_href", "rating"]], reader)

In [6]:
sim_options = {
    "name": ["msd", "cosine"],
    "min_support": [2, 3, 4, 5, 6, 7],
    "user_based": [False, True],
}

param_grid = {"sim_options": sim_options}

gs = GridSearchCV(KNNWithMeans, param_grid, measures=["rmse", "mae"], cv=5)
gs.fit(data)

print(gs.best_score["rmse"])
print(gs.best_params["rmse"])

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [7]:
param_grid = {
    "n_epochs": [5, 10],
    "lr_all": [0.002, 0.005],
    "reg_all": [0.4, 0.6]
}
gs = GridSearchCV(SVD, param_grid, measures=["rmse", "mae"], cv=3)

gs.fit(data)

print(gs.best_score["rmse"])
print(gs.best_params["rmse"])


1.1997534113196509
{'n_epochs': 10, 'lr_all': 0.005, 'reg_all': 0.6}


In [8]:
param_grid = {
    "n_epochs": [5, 10, 15, 20],
    "lr_all": [0.001, 0.002, 0.005, 0.01, 0.015, 0.02, 0.03, 0.04, 0.05],
    "reg_all": [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
}
gs = GridSearchCV(SVD, param_grid, measures=["rmse", "mae"], cv=5)

gs.fit(data)

print(gs.best_score["rmse"])
print(gs.best_params["rmse"])

KeyboardInterrupt: 

### Train model with tuned hyperparams

In [ ]:
# Train the SVD model
trainset = data.build_full_trainset()

# Define hyperparameters as a dictionary
hyperparams = {
    #'n_factors': 10,
    'n_epochs': 5,
    'lr_all': 0.15,
    'reg_all': 0.5
}

# Train the SVD model with specified hyperparameters
model = SVD(**hyperparams)
model.fit(trainset)

### Check top predictions for individual user

In [ ]:
df.username.value_counts()

In [ ]:
username = "Chris"
user_ratings_df = df[df["username"] == username]
user_ratings_df

In [ ]:
# List of products to predict ratings for
products_to_predict = list(set(df.product_href) - set(user_ratings_df.product_href))

predictions = []
for item_id in products_to_predict:
    prediction = model.predict(username, item_id)
    predictions.append((item_id, prediction.est))

# Sort predictions by estimated rating in descending order
predictions.sort(key=lambda x: x[1], reverse=True)

# Get the top 10 highest-rated predictions
top_10_predictions = predictions[:10]

# Print the top 10 highest-rated predictions
print("Top 10 Highest-Rated Predictions:\n")
recomendations = []
for product_href, rating in top_10_predictions:
    recomendations.append(product_href)
    print(f"Item {product_href}: Predicted rating = {rating}\n")

### Manually inspect 10 recommendations

In [ ]:
product_df = pd.read_csv("data/cleaned_wine_df.csv")
product_df[product_df["href"].isin(recomendations)]

In [ ]:
## Test using mlflow

param_grid = {
    "n_epochs": [5, 10],
    "lr_all": [0.001, 0.04, 0.05],
    "reg_all": [0.2, 0.3]
}
gs = GridSearchCV(SVD, param_grid, measures=["rmse", "mae"], cv=5)

# Start MLflow run
experiment_name = "test-ex"
mlflow.set_experiment(experiment_name)
mlflow.set_tracking_uri("mlflow/")
with mlflow.start_run():

    mlflow.sklearn.autolog()
    
    # Perform grid search
    gs.fit(data)

    # Log best score and parameters to MLflow
    mlflow.log_metric("best_rmse", gs.best_score["rmse"])
    #mlflow.log_metric("best_mae", gs.best_score["mae"])
    #mlflow.log_params("best_rmse_params", gs.best_params["rmse"])
    #mlflow.log_params("best_mae_params", gs.best_params["mae"])


print(gs.best_score["rmse"])
print(gs.best_params["rmse"])
print(gs.best_score["mae"])
print(gs.best_params["mae"])

In [3]:
# Start MLflow run
mlflow.set_experiment("test-ex")
with mlflow.start_run():

    mlflow.log_metrics({"mse": 0.69})

2024/05/10 21:02:52 INFO mlflow.tracking.fluent: Experiment with name 'test-ex' does not exist. Creating a new experiment.
